# Приступаем к работе над моделью детектинга  

Возьмём предобученную модель Yolo (v10), размеченный датасет, предобработаем его и обучим.

In [1]:
import os
import cv2
import time

from ultralytics import YOLO

### Подключаем Torch (Cuda)  

Данная версия Torch поддерживает соединение с cuda 13.1

In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

import torchvision
print(torchvision.__version__) 

2.9.1+cu130
True
0.24.1+cu130


# Очистка неразмеченных данных  

In [4]:
from pathlib import Path

images_path = Path('D://AIM/AI-Flow-Detecting/ai-core/dataset/train')

for img_file in images_path.glob("*.jpg"):
    txt_file = img_file.with_suffix('.txt')
    
    # Проверка: нет файла .txt ИЛИ он пустой (включая только пробелы/переносы)
    if not txt_file.exists() or not txt_file.read_text(encoding='utf-8').strip():
        img_file.unlink()  # Удалить .jpg
        if txt_file.exists():
            txt_file.unlink()  # Опционально: удалить и пустой .txt
        print(f"Удалено изображение (и, возможно, пустой .txt): {img_file}")


Удалено изображение (и, возможно, пустой .txt): D:\AIM\AI-Flow-Detecting\ai-core\dataset\train\event_20251203_125431_786925.jpg
Удалено изображение (и, возможно, пустой .txt): D:\AIM\AI-Flow-Detecting\ai-core\dataset\train\event_20251203_125615_411830.jpg
Удалено изображение (и, возможно, пустой .txt): D:\AIM\AI-Flow-Detecting\ai-core\dataset\train\event_20251203_125620_074931.jpg
Удалено изображение (и, возможно, пустой .txt): D:\AIM\AI-Flow-Detecting\ai-core\dataset\train\event_20251203_125650_081464.jpg
Удалено изображение (и, возможно, пустой .txt): D:\AIM\AI-Flow-Detecting\ai-core\dataset\train\event_20251203_173005_846418.jpg


In [5]:
# PATHS
RAW_DATASET = Path("dataset/train")
PREPROCESSED_DATASET = Path("dataset/train_preprocessed")

PREPROCESSED_DATASET.mkdir(parents=True, exist_ok=True)

## Подробная предобработка зависящая от разных условий на кадре  

In [6]:
import cv2
import numpy as np
from pathlib import Path
import shutil
from enum import Enum

class SceneType(Enum):
    NIGHT = "night"
    TWILIGHT = "twilight"
    DAY = "day"
    FOGGY = "foggy"
    SNOW = "snow"
    RAIN = "rain"

# =========================
# ADVANCED SCENE DETECTION
# =========================

def detect_scene(img):
    """Улучшенная детекция сцены с детекцией дождя"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    mean_brightness = gray.mean()
    std_brightness = gray.std()
    saturation = hsv[:, :, 1].mean()
    
    # Детекция дождя (высокая частота + низкий контраст)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if laplacian_var > 100 and std_brightness < 40 and mean_brightness > 60:
        return SceneType.RAIN
    
    # Детекция тумана/снега
    if std_brightness < 35 and mean_brightness > 100:
        if mean_brightness > 140 and saturation < 40:
            return SceneType.SNOW
        return SceneType.FOGGY
    
    # Время суток
    if mean_brightness < 70:
        return SceneType.NIGHT
    elif mean_brightness < 110:
        return SceneType.TWILIGHT
    else:
        return SceneType.DAY

# =========================
# CORE ENHANCEMENTS
# =========================

def adaptive_gamma(img, scene):
    """Адаптивная гамма-коррекция"""
    gamma_map = {
        SceneType.NIGHT: 1.4,
        SceneType.TWILIGHT: 1.15,
        SceneType.DAY: 1.0,
        SceneType.FOGGY: 1.12,
        SceneType.SNOW: 0.97,
        SceneType.RAIN: 1.1
    }
    
    gamma = gamma_map[scene]
    inv = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv) * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(img, table)

def dual_clahe(img, scene):
    """Двойная CLAHE: LAB + RGB для лучшей детализации"""
    # CLAHE в LAB (яркость)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    
    params = {
        SceneType.NIGHT: (2.2, (8, 8)),
        SceneType.TWILIGHT: (1.8, (10, 10)),
        SceneType.DAY: (1.3, (12, 12)),
        SceneType.FOGGY: (2.5, (8, 8)),
        SceneType.SNOW: (1.8, (10, 10)),
        SceneType.RAIN: (2.0, (8, 8))
    }
    
    clip_limit, tile_size = params[scene]
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    l = clahe.apply(l)
    
    lab = cv2.merge((l, a, b))
    result = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    
    # Лёгкая CLAHE на каждом канале RGB (для цветовой детализации)
    if scene in [SceneType.NIGHT, SceneType.FOGGY, SceneType.RAIN]:
        clahe_light = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(10, 10))
        b_ch, g_ch, r_ch = cv2.split(result)
        b_ch = clahe_light.apply(b_ch)
        g_ch = clahe_light.apply(g_ch)
        r_ch = clahe_light.apply(r_ch)
        result = cv2.merge([b_ch, g_ch, r_ch])
    
    return result

def retinex_enhancement(img, scene):
    if scene not in [SceneType.NIGHT, SceneType.FOGGY, SceneType.RAIN]:
        return img
    
    # Конвертация в float
    img_float = img.astype(np.float32) + 1.0
    
    # Multi-scale gaussian blur
    scales = [15, 80, 250]
    weights = [1/3, 1/3, 1/3]
    
    retinex = np.zeros_like(img_float)
    
    for scale, weight in zip(scales, weights):
        blur = cv2.GaussianBlur(img_float, (0, 0), scale)
        retinex += weight * (np.log10(img_float) - np.log10(blur))
    
    # Нормализация
    retinex = (retinex - retinex.min()) / (retinex.max() - retinex.min()) * 255
    result = retinex.astype(np.uint8)
    
    # Смешивание с оригиналом
    alpha = 0.3 if scene == SceneType.NIGHT else 0.2
    return cv2.addWeighted(result, alpha, img, 1-alpha, 0)

def advanced_dehazing(img, scene):
    """Продвинутый dehazing для тумана/снега/дождя"""
    if scene not in [SceneType.FOGGY, SceneType.SNOW, SceneType.RAIN]:
        return img
    
    # Guided filter dehazing
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Dark channel
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    dark = cv2.erode(cv2.min(img[:,:,0], cv2.min(img[:,:,1], img[:,:,2])), kernel)
    
    # Atmospheric light
    flat_dark = dark.flatten()
    flat_img = img.reshape(-1, 3)
    indices = np.argsort(flat_dark)[-int(0.001 * len(flat_dark)):]
    atmospheric_light = np.mean(flat_img[indices], axis=0)
    
    # Transmission estimation
    norm = img / (atmospheric_light + 1e-6)
    trans = 1 - 0.8 * cv2.erode(
        cv2.min(norm[:,:,0], cv2.min(norm[:,:,1], norm[:,:,2])), 
        kernel
    )
    trans = np.maximum(trans, 0.15)
    
    # Recover
    result = np.zeros_like(img, dtype=np.float32)
    for c in range(3):
        result[:,:,c] = (img[:,:,c] - atmospheric_light[c]) / (trans + 1e-6) + atmospheric_light[c]
    
    result = np.clip(result, 0, 255).astype(np.uint8)
    
    # Blend with original
    blend_factor = 0.6 if scene == SceneType.RAIN else 0.7
    return cv2.addWeighted(result, blend_factor, img, 1-blend_factor, 0)

def adaptive_unsharp_mask(img, scene):
    """Адаптивный Unsharp Mask вместо простой резкости"""
    # Параметры по сцене
    params = {
        SceneType.NIGHT: (5, 1.3, 0.8),      # radius, amount, threshold
        SceneType.TWILIGHT: (5, 1.2, 0.6),
        SceneType.DAY: (3, 1.1, 0.5),
        SceneType.FOGGY: (5, 1.4, 1.0),
        SceneType.SNOW: (3, 1.15, 0.7),
        SceneType.RAIN: (5, 1.3, 0.9)
    }
    
    radius, amount, threshold = params[scene]
    
    # Gaussian blur
    blurred = cv2.GaussianBlur(img, (0, 0), radius)
    
    # Unsharp mask
    sharpened = cv2.addWeighted(img, amount, blurred, -(amount-1), 0)
    
    # Threshold (только усиливаем края, не шум)
    diff = cv2.absdiff(img, blurred)
    mask = cv2.threshold(cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY), 
                         threshold, 255, cv2.THRESH_BINARY)[1]
    mask = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR) / 255.0
    
    return (sharpened * mask + img * (1 - mask)).astype(np.uint8)

def edge_aware_sharpening(img, scene):
    """Усиление краёв с сохранением гладких областей"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Детекция краёв
    edges = cv2.Canny(gray, 50, 150)
    edges = cv2.GaussianBlur(edges, (3, 3), 1)
    edges = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=1)
    
    # Создание маски краёв
    edge_mask = edges.astype(np.float32) / 255.0
    edge_mask = cv2.GaussianBlur(edge_mask, (5, 5), 2)
    edge_mask = np.expand_dims(edge_mask, axis=2)
    
    # Резкость только на краях
    strength_map = {
        SceneType.NIGHT: 0.15,
        SceneType.TWILIGHT: 0.10,
        SceneType.DAY: 0.07,
        SceneType.FOGGY: 0.18,
        SceneType.SNOW: 0.10,
        SceneType.RAIN: 0.15
    }
    
    strength = strength_map[scene]
    
    laplacian = cv2.Laplacian(img, cv2.CV_64F)
    laplacian = cv2.convertScaleAbs(laplacian)
    
    # Применяем только на краях
    enhanced = img.astype(np.float32) + laplacian.astype(np.float32) * edge_mask * strength
    return np.clip(enhanced, 0, 255).astype(np.uint8)

def bilateral_detail_enhance(img, scene):
    """Bilateral фильтр для усиления деталей без шума"""
    if scene not in [SceneType.NIGHT, SceneType.RAIN]:
        return img
    
    # Bilateral filter сохраняет края
    smooth = cv2.bilateralFilter(img, 5, 50, 50)
    
    # Усиливаем детали
    details = cv2.subtract(img, smooth)
    details_enhanced = cv2.multiply(details, np.array([1.5]))
    
    result = cv2.add(smooth, details_enhanced)
    return np.clip(result, 0, 255).astype(np.uint8)

def color_balance(img, scene):
    """Автоматическая цветовая коррекция"""
    if scene == SceneType.DAY:
        return img
    
    # Простой white balance
    result = img.copy().astype(np.float32)
    
    for i in range(3):
        channel = result[:, :, i]
        avg = channel.mean()
        result[:, :, i] = channel * (128.0 / (avg + 1e-6))
    
    result = np.clip(result, 0, 255).astype(np.uint8)
    
    # Blend
    alpha = 0.3 if scene == SceneType.NIGHT else 0.2
    return cv2.addWeighted(result, alpha, img, 1-alpha, 0)

# =========================
# PIPELINES
# =========================

def premium_pipeline(img):
    """ПРЕМИУМ пайплайн - максимальное качество"""
    scene = detect_scene(img)
    
    # Цветовая коррекция
    img = color_balance(img, scene)
    
    # Dehazing (если нужно)
    img = advanced_dehazing(img, scene)
    
    # Retinex (для сложных условий)
    img = retinex_enhancement(img, scene)
    
    # Гамма
    img = adaptive_gamma(img, scene)
    
    # Dual CLAHE
    img = dual_clahe(img, scene)
    
    # Bilateral detail enhancement
    img = bilateral_detail_enhance(img, scene)
    
    # Edge-aware sharpening
    img = edge_aware_sharpening(img, scene)
    
    # Unsharp mask
    img = adaptive_unsharp_mask(img, scene)
    
    return img, scene

def balanced_pipeline(img):
    """СБАЛАНСИРОВАННЫЙ пайплайн - оптимум"""
    scene = detect_scene(img)
    
    # Легкий dehazing
    img = advanced_dehazing(img, scene)
    
    # Гамма
    img = adaptive_gamma(img, scene)
    
    # Dual CLAHE (ключевое улучшение)
    img = dual_clahe(img, scene)
    
    # Retinex для ночи/тумана
    if scene in [SceneType.NIGHT, SceneType.FOGGY]:
        img = retinex_enhancement(img, scene)
    
    # Edge-aware sharpening
    img = edge_aware_sharpening(img, scene)
    
    # Unsharp mask
    img = adaptive_unsharp_mask(img, scene)
    
    return img, scene

def minimal_pipeline(img):
    scene = detect_scene(img)
    
    # Только CLAHE + легкая резкость
    img = dual_clahe(img, scene)
    img = edge_aware_sharpening(img, scene)
    
    return img, scene

# =========================
# DATASET PREPROCESSING
# =========================

def preprocess_dataset(raw_path, output_path, mode='balanced', 
                       save_stats=True, preview_samples=5):
    """Предобработка датасета с расширенными возможностями"""
    RAW_DATASET = Path(raw_path)
    PREPROCESSED_DATASET = Path(output_path)
    PREPROCESSED_DATASET.mkdir(parents=True, exist_ok=True)
    
    # Выбор пайплайна
    pipelines = {
        'premium': premium_pipeline,
        'balanced': balanced_pipeline,
        'minimal': minimal_pipeline
    }
    pipeline_func = pipelines.get(mode, balanced_pipeline)
    
    print(f"{'='*70}")
    print(f"YOLO Preprocessing Pipeline v2.0")
    print(f"{'='*70}")
    print(f"Входные данные: {raw_path}")
    print(f"Выходная папка: {output_path}")
    print(f"Режим: {mode.upper()}")
    print(f"{'='*70}\n")
    
    # Сбор файлов
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
    image_files = []
    for ext in image_extensions:
        image_files.extend(RAW_DATASET.glob(ext))
    
    print(f"Найдено изображений: {len(image_files)}\n")
    
    if len(image_files) == 0:
        print("Изображения не найдены!")
        return
    
    # Статистика
    stats = {scene.value: 0 for scene in SceneType}
    total = 0
    errors = 0
    preview_saved = 0
    
    # Создаём папку для preview
    preview_dir = PREPROCESSED_DATASET / "preview_comparison"
    if preview_samples > 0:
        preview_dir.mkdir(exist_ok=True)
    
    # Обработка
    for idx, img_file in enumerate(image_files):
        try:
            txt_file = img_file.with_suffix(".txt")
            if not txt_file.exists():
                continue
            
            # Чтение
            img_original = cv2.imread(str(img_file))
            if img_original is None:
                errors += 1
                continue
            
            # Обработка
            processed, scene = pipeline_func(img_original.copy())
            stats[scene.value] += 1
            
            # Сохранение
            cv2.imwrite(str(PREPROCESSED_DATASET / img_file.name), processed)
            shutil.copy(txt_file, PREPROCESSED_DATASET / txt_file.name)
            
            # Preview для первых N изображений
            if preview_samples > 0 and preview_saved < preview_samples:
                h, w = img_original.shape[:2]
                comparison = np.zeros((h, w*2, 3), dtype=np.uint8)
                comparison[:, :w] = img_original
                comparison[:, w:] = processed
                
                font = cv2.FONT_HERSHEY_SIMPLEX
                cv2.putText(comparison, 'ORIGINAL', (10, 30), font, 1, (0, 255, 0), 2)
                cv2.putText(comparison, f'PROCESSED ({scene.value})', (w+10, 30), font, 1, (0, 255, 0), 2)
                
                cv2.imwrite(str(preview_dir / f"preview_{idx+1}.jpg"), comparison)
                preview_saved += 1
            
            total += 1
            if total % 100 == 0:
                print(f"{total}/{len(image_files)} ({total/len(image_files)*100:.1f}%)")
                
        except Exception as e:
            print(f"{img_file.name}: {str(e)}")
            errors += 1
    
    # Итоговая статистика
    print(f"\n{'='*70}")
    print(f"ОБРАБОТКА ЗАВЕРШЕНА")
    print(f"{'='*70}")
    print(f"Статистика:")
    print(f"  • Успешно обработано: {total}")
    print(f"  • Ошибок: {errors}")
    print(f"  • Процент успеха: {total/(total+errors)*100:.1f}%")
    
    print(f"\nРаспределение по сценам:")
    for scene, count in sorted(stats.items(), key=lambda x: -x[1]):
        if count > 0:
            bar_length = int(count / total * 40)
            bar = "█" * bar_length + "░" * (40 - bar_length)
            print(f"  {scene.upper():12} {bar} {count:5} ({count/total*100:5.1f}%)")
    
    if preview_samples > 0:
        print(f"\n Preview: {preview_dir}")
    
    # Сохранение статистики
    if save_stats:
        stats_file = PREPROCESSED_DATASET / "preprocessing_stats.txt"
        with open(stats_file, 'w') as f:
            f.write(f"Preprocessing Mode: {mode}\n")
            f.write(f"Total: {total}\n")
            f.write(f"Errors: {errors}\n")
            for scene, count in stats.items():
                f.write(f"{scene}: {count} ({count/total*100:.1f}%)\n")
        print(f"Статистика сохранена: {stats_file}")
    
    print(f"{'='*70}\n")


preprocess_dataset(
    raw_path="dataset/train",
    output_path="dataset/train_preprocessed",
    mode='balanced',
    save_stats=True
)

YOLO Preprocessing Pipeline v2.0
Входные данные: dataset/train
Выходная папка: dataset/train_preprocessed
Режим: BALANCED

Найдено изображений: 834

100/834 (12.0%)
200/834 (24.0%)
300/834 (36.0%)
400/834 (48.0%)
500/834 (60.0%)
600/834 (71.9%)
700/834 (83.9%)
800/834 (95.9%)

ОБРАБОТКА ЗАВЕРШЕНА
Статистика:
  • Успешно обработано: 834
  • Ошибок: 0
  • Процент успеха: 100.0%

Распределение по сценам:
  RAIN         ████████████████░░░░░░░░░░░░░░░░░░░░░░░░   352 ( 42.2%)
  DAY          ███████████████░░░░░░░░░░░░░░░░░░░░░░░░░   316 ( 37.9%)
  TWILIGHT     ███████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░   166 ( 19.9%)

 Preview: dataset\train_preprocessed\preview_comparison
Статистика сохранена: dataset\train_preprocessed\preprocessing_stats.txt



In [7]:
from sklearn.model_selection import train_test_split

dataset_path = "dataset/train_preprocessed"
train_dest = "dataset/train_split"
val_dest = "dataset/val_split"

val_ratio = 0.2
random_seed = 42

Path(train_dest).mkdir(parents=True, exist_ok=True)
Path(val_dest).mkdir(parents=True, exist_ok=True)

all_files = [f for f in os.listdir(dataset_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
print(f"Найдено {len(all_files)} изображений")

train_files, val_files = train_test_split(
    all_files,
    test_size=val_ratio,
    random_state=random_seed
)

def copy_files(files, destination):
    for file in files:
        base = os.path.splitext(file)[0]
        shutil.copy(os.path.join(dataset_path, file), os.path.join(destination, file))
        txt = f"{base}.txt"
        src_txt = os.path.join(dataset_path, txt)
        if os.path.exists(src_txt):
            shutil.copy(src_txt, os.path.join(destination, txt))

copy_files(train_files, train_dest)
copy_files(val_files, val_dest)

print("Train / Val split готов")

Найдено 417 изображений
Train / Val split готов


### Обучение модели 

In [8]:
import torch
from ultralytics import YOLO

MODEL_PATH = 'yolo11m.pt'
OUTPUT_NAME = 'detect_finetuned'
PROJECT_DIR = 'runs/train'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используется: {device}")

# Автоматическое включение AMP только на GPU
use_amp = True if device == 'cuda' else False

model = YOLO(MODEL_PATH).to(device)

model.train(
    data='data.yaml',
    epochs=45,
    imgsz=768,
    batch=16,
    device=device,
    name=OUTPUT_NAME,
    project=PROJECT_DIR,

    augment=True,
    optimizer='AdamW',

    workers=4,
    amp=use_amp,

    hsv_h=0.03,
    hsv_s=0.7,
    hsv_v=0.8,

    translate=0.3,
    scale=0.5,
    fliplr=0.5,
    mosaic=0.8,
    mixup=0.1,
    degrees=10.0,

    lr0=3e-4,
    lrf=0.01,

    rect=False,
    cache='ram',
    plots=True,
    save=True,
)

Используется: cuda
New https://pypi.org/project/ultralytics/8.3.252 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.250  Python-3.14.2 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=45, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.03, hsv_s=0.7, hsv_v=0.8, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=0.8, multi_scale=False, name=detect_finetuned6,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001BA31752F90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [9]:
model.save('runs/result/preprocessdetect.pt')
print("✅ Модель дообучена и сохранена.")

✅ Модель дообучена и сохранена.


In [10]:
metrics = model.val(
    data="data.yaml",
    imgsz=640,
    conf=0.25
)

Ultralytics 8.3.250  Python-3.14.2 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access  (ping: 0.10.2 ms, read: 81.827.5 MB/s, size: 704.7 KB)
val: Scanning D:\AIM\AI-Flow-Detecting\ai-core\dataset\val_split.cache... 84 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 84/84 120.9Kit/s 0.0s
val: D:\AIM\AI-Flow-Detecting\ai-core\dataset\val_split\20250810_153130.jpg: 2 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.0s/it 6.2s0.3s1
                   all         84        516      0.693      0.547      0.619       0.29
Speed: 3.1ms preprocess, 11.6ms inference, 0.0ms loss, 3.2ms postprocess per image
Results saved to D:\AIM\AI-Flow-Detecting\runs\detect\val


In [15]:
import signal
from ultralytics import YOLO
from collections import defaultdict

# GRACEFUL SHUTDOWN
running = True

def signal_handler(sig, frame):
    global running
    print("\nОстановка...")
    running = False

signal.signal(signal.SIGINT, signal_handler)

# MODEL
print("Загрузка модели...")
model = YOLO("runs/result/preprocessdetect.pt")

# VIDEO SOURCE (ССЫЛКА!)
SOURCE = "https://restreamer.vms.evo73.ru/918335436b92ac26/stream.m3u8"

cap = cv2.VideoCapture(SOURCE)
if not cap.isOpened():
    raise RuntimeError("❌ Не удалось открыть видеопоток")

# =========================
# TRACK STORAGE
# =========================
tracks = defaultdict(list)

print("▶️ Старт обработки. Нажми Q или Ctrl+C")

frame_id = 0

while running and cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("⚠️ Потеря кадра, повтор...")
        cv2.waitKey(500)
        continue

    frame_id += 1

    # =========================
    # YOLO + TRACKING
    # =========================
    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.4,
        iou=0.4,
        classes=[0],  # человек
        verbose=False
    )

    if results and results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy()

        for box, track_id in zip(boxes, ids):
            x1, y1, x2, y2 = map(int, box)

            # центр bbox
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2

            tracks[track_id].append((cx, cy))

            # =========================
            # BEHAVIOR ANALYSIS
            # =========================
            state = "unknown"

            if len(tracks[track_id]) >= 2:
                dx = tracks[track_id][-1][0] - tracks[track_id][-2][0]
                dy = tracks[track_id][-1][1] - tracks[track_id][-2][1]
                speed = np.sqrt(dx*dx + dy*dy)

                if speed < 2:
                    state = "standing"
                    color = (0, 0, 255)
                else:
                    state = "moving"
                    color = (0, 255, 0)
            else:
                color = (255, 255, 0)

            # =========================
            # DRAW
            # =========================
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(
                frame,
                f"ID {int(track_id)} | {state}",
                (x1, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

    cv2.putText(
        frame,
        f"Frame: {frame_id}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    cv2.imshow("YOLO + Tracking + Behavior", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        running = False

# =========================
# CLEANUP
# =========================
cap.release()
cv2.destroyAllWindows()
print("✅ Завершено")


Загрузка модели...
▶️ Старт обработки. Нажми Q или Ctrl+C
✅ Завершено


In [14]:
import cv2
from ultralytics import YOLO
import numpy as np
import signal
import sys

# Глобальная переменная для graceful shutdown
running = True

def signal_handler(sig, frame):
    """Обработчик сигнала Ctrl+C"""
    global running
    print("\nПолучен сигнал остановки...")
    running = False

# Регистрация обработчика сигнала
signal.signal(signal.SIGINT, signal_handler)

def dice_coef(gt_mask, pred_mask):
    """
    Коэффициент Дайса для сравнения масок
    """
    gt = gt_mask.astype(bool)
    pr = pred_mask.astype(bool)
    inter = (gt & pr).sum()
    return 2 * inter / (gt.sum() + pr.sum() + 1e-6)

def process_video_with_tracking(model, source, tracker='bytetrack.yaml', conf=0.5, show=True, save=False):
    global running
    
    # Открыть источник видео (видеофайл или поток)
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise Exception(f"Ошибка: Не удалось открыть источник: {source}")
    
    # Получить параметры видео
    fps = int(cap.get(cv2.CAP_PROP_FPS)) if source.endswith(('.mp4', '.avi', '.mov')) else 30
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if save:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        output_path = "output_video.mp4"
        out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    frame_count = 0
    print(f"Начало обработки. Нажмите 'q' в окне видео или Ctrl+C в консоли для остановки.")

    while running:
        ret, frame = cap.read()

        if not ret:
            print("Не удалось получить кадр. Повторная попытка...")
            cv2.waitKey(1000)
            continue
        
        frame_count += 1
        
        try:
            print(f"Обрабатываем кадр {frame_count}")
            
            # Преобразуем conf в обычный float
            conf = float(conf)
            
            results = model.track(
                frame,
                classes=[0],  
                iou=0.4,      
                conf=conf,     
                persist=True,  
                imgsz=640,     
                verbose=False, 
                tracker=tracker    
            )

            annotated_frame = frame.copy()
            people_count = 0

            if len(results) > 0 and results[0].boxes is not None:
                boxes = results[0].boxes.xyxy.cpu().numpy()
                confidences = results[0].boxes.conf.cpu().numpy()
                people_count = len(boxes)
                
                for i, (box, conf) in enumerate(zip(boxes, confidences)):
                    x1, y1, x2, y2 = map(int, box)
                    color = (0, 255, 0)
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
                    label = f'Человек {conf:.2f}'
                    cv2.putText(annotated_frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            cv2.putText(annotated_frame, f'Shot: {frame_count}', (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.putText(annotated_frame, f'Peoples: {people_count}', (10, 60),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            if save:
                out.write(annotated_frame)

            if show:
                display_frame = cv2.resize(annotated_frame, (0, 0), fx=0.75, fy=0.75)
                cv2.imshow("YOLOv — Обнаружение людей", display_frame)
                key = cv2.waitKey(1) & 0xFF
                if key == ord("q"):
                    running = False
                elif key == 27:  # ESC
                    running = False

        except Exception as e:
            print(f"Ошибка при обработке кадра {frame_count}: {e}")
            continue

    cap.release()
    if save:
        out.release()

    cv2.destroyAllWindows()
    print(f"Обработка завершена. Обработано кадров: {frame_count}")



    # Освобождение ресурсов
    cap.release()
    if save:
        out.release()

    # Закрытие всех окон OpenCV
    cv2.destroyAllWindows()
    print(f"Обработка завершена. Обработано кадров: {frame_count}")

# Пример использования:
if __name__ == "__main__":
    # Загрузка модели
    print("Загрузка модели...")
    model = YOLO('runs/result/preprocessdetect.pt')

    # Для видеопотока
    source = "https://restreamer.vms.evo73.ru/918335436b92ac26/stream.m3u8  "

    try:
        # Запуск обработки видео
        process_video_with_tracking(
            model,
            source=source.strip(),
            tracker='bytetrack.yaml',
            conf=0.5,
            show=True,
            save=False
        )
    except Exception as e:
        print(f"Ошибка: {e}")
    finally:
        print("Программа завершена.")

Загрузка модели...
Начало обработки. Нажмите 'q' в окне видео или Ctrl+C в консоли для остановки.
Обрабатываем кадр 1
Обрабатываем кадр 2
Обрабатываем кадр 3
Обрабатываем кадр 4
Обрабатываем кадр 5
Обрабатываем кадр 6
Обрабатываем кадр 7
Обрабатываем кадр 8
Обрабатываем кадр 9
Обрабатываем кадр 10
Обрабатываем кадр 11
Обрабатываем кадр 12
Обрабатываем кадр 13
Обрабатываем кадр 14
Обрабатываем кадр 15
Обрабатываем кадр 16
Обрабатываем кадр 17
Обрабатываем кадр 18
Обрабатываем кадр 19
Обрабатываем кадр 20
Обрабатываем кадр 21
Обрабатываем кадр 22
Обрабатываем кадр 23
Обрабатываем кадр 24
Обрабатываем кадр 25
Обрабатываем кадр 26
Обрабатываем кадр 27
Обрабатываем кадр 28
Обрабатываем кадр 29
Обрабатываем кадр 30
Обрабатываем кадр 31
Обрабатываем кадр 32
Обрабатываем кадр 33
Обрабатываем кадр 34
Обрабатываем кадр 35
Обрабатываем кадр 36
Обрабатываем кадр 37
Обрабатываем кадр 38
Обрабатываем кадр 39
Обрабатываем кадр 40
Обрабатываем кадр 41
Обрабатываем кадр 42
Обрабатываем кадр 43
Обрабаты